# 从零开始构建推理引擎（一）：PyTorch 复现 Qwen3 推理过程

## 0. 准备虚拟环境

本文使用的 Python 版本为 3.11。读者可以使用自己青睐的虚拟环境工具，如 conda 或 virtualenv 或 uv 来创建本 notebook 所需的虚拟环境。

本文为读者提供了 `pyproject.toml` 描述虚拟环境所依赖的安装包，你可以使用 `uv sync` 或者其他方式来使用它。

## 1. 模型资源

### 1.1 下载模型资源

要实现某个模型的推理，首先需要获得模型的资源。
各个模型在开源的时候，一般都会在 huggingface 上发布自己的模型资源。
本文以 Qwen3-0.6B 为例，你可以在 [Qwen/Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B) 上找到模型资源。
模型资源一般使用 huggingface 提供的 [CLI](https://huggingface.co/docs/huggingface_hub/guides/cli) 来下载。

请首先按照 CLI 文档安装 `hf` 并下载 Qwen3-0.6B 的模型资源。
为了便于查看模型资源的内容，这次我们将模型资源下载到 `~/huggingface/Qwen3-0.6B/` 中。

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "true"  # 取消 huggingface 检测到进程 fork 的警告

In [ ]:
!hf download Qwen/Qwen3-0.6B --local-dir ~/huggingface/Qwen3-0.6B/

### 1.2 模型资源介绍

我们需要了解一下模型资源的构成，明确推理过程会用到哪些资源文件。

#### 1.2.1 模型配置 `config.json`

模型配置文件 `config.json` 中记录了模型参数，比如词表大小、隐藏层维度、层数等。
发布的模型共享同一套代码，但可以有好几种变体和规格。
这些参数便是在描述模型规格的，用于在推理中初始化模型对象。

In [ ]:
!cat ~/huggingface/Qwen3-0.6B/config.json

#### 1.2.2 模型权重 `model.safetensors`

对于 Qwen3-0.6B 而言，所有层的权重都保存在 `model.safetensors` 文件中。
对于一些更大的模型，权重会保存在多个文件中。

模型权重文件可以通过 `safetensors` 库打开，我们可粗浅看一下这些权重的结构。
打开后的内容是一个字典，键为层名称，值为该层的权重张量。

In [ ]:
import os
from safetensors import safe_open

path = os.path.expanduser("~/huggingface/Qwen3-0.6B/model.safetensors")

tensors = {}

with safe_open(path, framework="pt", device="cpu") as f:
    for key in f.keys()[:10]:
        tensors[key] = f.get_tensor(key)
        print(key, tensors[key].shape)

#### 1.2.3 分词器配置 `tokenizer.json`

模型使用的分词器配置，用于将输入文本转换为模型可处理的输入。
举个例子，分词器将文本 `hello\w` 转换为 `hel` 和 `lo\w` 两个 token，分别映射到数字 123 和 987。
推理引擎后续可以使用 `[123, 987]` 作为模型的输入。

Qwen3 模型使用的分词是 Byte-Pair Encoding（BPE），是一种基于统计的方法，
将文本中的单词或字符进行编码，并生成一个编码表，输出到 `tokenizer.json` 文件中。

另外还有一个 `tokenizer_config.json` 文件，用于配置分词器的参数，以及标记一些特殊 token。
根据 `tokenizer_config.json` 可以看到 transformers 库所使用的分词器类是 `Qwen2Tokenizer`。

实际上除了 transformers 库以外也存在多种分词库，有兴趣的读者还可以根据 BPE 算法和 `tokenizer.json` 自行实现一个分词器，并优化分词器的性能。
这里我们简单使用 transformers 库中的分词器即可，下面是使用分词器的一个简单例子。

In [ ]:
import os
from transformers import Qwen2Tokenizer

path = os.path.expanduser("~/huggingface/Qwen3-0.6B")

tokenizer = Qwen2Tokenizer.from_pretrained(path)
tokenizer.padding_side = "left"  # 推理需要左侧填充

# 用户原始的输入
prompts = ["Hello, how are you?", "Hello"]
# 通过 chat_template 函数填充用户的输入，具体可以见 tokenizer_config.json 中对应的脚本
chats = [[{"role": "user", "content": prompt}] for prompt in prompts]  # 两组多轮对话
texts = [
    tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    for messages in chats
]
print("-" * 40)
print(texts[0])
print("-" * 40)
# 对输入进行编码，生成 token id 张量，注意需要 padding
# 形状为 [batch_size, seq_len]
model_inputs = tokenizer([text for text in texts], return_tensors="pt", padding=True)
for k in model_inputs.keys():
    print(k)
    print(model_inputs[k])
    print(model_inputs[k].shape)
# 我们还可以对这些 token id 张量进行解码
print("-" * 40)
print(tokenizer.decode(model_inputs.input_ids[0]))
print("-" * 40)

另外旧版本的分词器所使用的是 `merges.txt` 和 `vocab.json`，模型资源中会给出但不一定会用到。
这两个文件所包含的内容与 `tokenizer.json` 是等价的。

#### 1.2.4 生成配置 `generation_config.json`

最后是生成配置，用于控制生成过程。
其中记录了生成过程中使用的参数，如温度、TopK、TopP 等。
在编写生成 token 的逻辑时，会用到这些内容。

生成配置文件 `generation_config.json` 的内容如下：

In [ ]:
!cat ~/huggingface/Qwen3-0.6B/generation_config.json

## 2. 使用 Transformers 进行推理

为了实现 Qwen3 的推理，除了准备好前面提到的模型资源以外，还需要准备模型推理代码。
模型在 huggingface 上发布的时候，已经提供了相应的推理代码，有助于我们理解模型结构并移植到任意的推理引擎中。

要复现 Qwen3 的推理，我们实际上需要理解 Qwen3 的模型结构。
首先我们从输入输出来只管感受一下 Qwen3 的推理。
我们可以复制并修改 huggingface 上 Qwen3 的官方示例并运行。

In [ ]:
import os
import random

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 设置种子以达到可复现的推理
seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)

model_name = "~/huggingface/Qwen3-0.6B"

path = os.path.expanduser(model_name)

# 加载分词器和模型
tokenizer = AutoTokenizer.from_pretrained(path)
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    path,
    torch_dtype="auto",
    device_map="auto",
)

# 准备模型输入
prompts = [
    "Give me a short introduction to large language model.",
    "1 + 1 = ?",
]
chats = [[{"role": "user", "content": prompt}] for prompt in prompts]
texts = [
    tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    for messages in chats
]
model_inputs = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)

# 将模型输入传进去，得到输出
generated_ids_list = model.generate(
    **model_inputs,
    max_new_tokens=256,
    do_sample=False,
)

for i, generated_ids in enumerate(generated_ids_list):
    # 去掉提示词的部分，之保留模型生成的内容
    output_ids = generated_ids[len(model_inputs.input_ids[i]) :].tolist()

    # 分离出 thinking 的部分
    try:
        # 从右开始找到第一个 151668 (</think>) 也就是特殊的 thinking 标记
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    thinking_content = tokenizer.decode(
        output_ids[:index], skip_special_tokens=True
    ).strip("\n")
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

    print("-" * 40)
    print("thinking content:", thinking_content)
    print("content:", content)

## 3. 根据 Transformers 源码实现 Qwen3 模型

接下来我们根据 transformers 所实现的 Qwen3 模型来构建我们自己的 Qwen3 模型。
简单来说其实这里并不需要知道多少算法上的内容，但还是需要了解一下模型结构。

### 3.1 模型结构阅读

如果读者使用的是 uv 或者 virtualenv 管理的虚拟环境，可以打开
`.venv/lib/python3.11/site-packages/transformers/models/qwen3/modeling_qwen3.py` 来了解模型结构，
这份文件也可以在 transformers 的 github 仓库中找到。

这里是我的个人思考，我实现一个模型会从自底向上实现，主要是便于验证模型每一层、以及拼起来之后的正确性。
但在阅读模型结构的时候，则需要自顶向下阅读。下面是我阅读模型结构的笔记。

#### QWen3Model

首先抓住模型最外层的结构，即 `QWen3Model` 类。
另外要特别关心每一个类在 `forward` 方法处的输入和输出。

```python
class Qwen3Model(Qwen3PreTrainedModel):
    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values: Optional[Cache] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        use_cache: Optional[bool] = None,
        cache_position: Optional[torch.LongTensor] = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> BaseModelOutputWithPast:
    ...
```